# Functional Programming in Python

Functional programming treats computation as the evaluation of functions, favoring:

- Functions as **first-class objects** (assign to variables, pass as arguments, return from functions)
- **Pure functions** with no side effects, and avoiding mutable state where possible
- Building programs by **composing small functions** instead of long imperative sequences

Python supports this style through: `lambda`, `map`/`filter`, the `functools` and `operator`
modules, `itertools`, comprehensions, generators, and decorators — each covered below.

## 1. Functions as First-Class Objects

Functions can be assigned to variables and passed around like any other value.

In [1]:
def square(x):
    return x * x

# Assign a function to a variable
operation = square
print("operation(6):", operation(6))

# Pass a function as an argument
words = ["banana", "kiwi", "apple", "fig"]
print("Sorted by length:", sorted(words, key=len))

# lambda: a small anonymous function
cube = lambda x: x ** 3
print("cube(3):", cube(3))

operation(6): 36
Sorted by length: ['fig', 'kiwi', 'apple', 'banana']
cube(3): 27


## 2. `map()`, `filter()`, and `reduce()`

The three classic functional building blocks: transform every element (`map`), keep only some
elements (`filter`), and combine all elements into one value (`reduce`).

In [2]:
from functools import reduce

numbers = [1, 2, 3, 4, 5, 6]

doubled = list(map(lambda x: x * 2, numbers))
print("Doubled:", doubled)

evens = list(filter(lambda x: x % 2 == 0, numbers))
print("Evens:", evens)

total = reduce(lambda acc, x: acc + x, numbers)
print("Sum via reduce:", total)

product = reduce(lambda acc, x: acc * x, numbers)
print("Product via reduce:", product)

Doubled: [2, 4, 6, 8, 10, 12]
Evens: [2, 4, 6]
Sum via reduce: 21
Product via reduce: 720


## 3. `functools`

`functools.partial` pre-fills some arguments of a function to create a new, simpler function.
`functools.lru_cache` memoizes results so repeated calls with the same arguments are cached.

In [3]:
from functools import partial, lru_cache

def power(base, exponent):
    return base ** exponent

square_via_partial = partial(power, exponent=2)
cube_via_partial = partial(power, exponent=3)
print("square_via_partial(5):", square_via_partial(5))
print("cube_via_partial(5):", cube_via_partial(5))

@lru_cache(maxsize=None)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print("fibonacci(20):", fibonacci(20))
print("Cache info:", fibonacci.cache_info())

square_via_partial(5): 25
cube_via_partial(5): 125
fibonacci(20): 6765
Cache info: CacheInfo(hits=18, misses=21, maxsize=None, currsize=21)


## 4. `operator` Module

The `operator` module provides function equivalents of operators, avoiding small throwaway
lambdas like `lambda x, y: x + y`.

In [4]:
import operator

print("add(3, 4):", operator.add(3, 4))
print("mul(3, 4):", operator.mul(3, 4))
print("Sum via reduce + operator.add:", reduce(operator.add, numbers))

students = [("Alice", 88), ("Bob", 95), ("Charlie", 72)]

# itemgetter: sort tuples by index instead of a lambda
by_score = sorted(students, key=operator.itemgetter(1), reverse=True)
print("Sorted by score:", by_score)

add(3, 4): 7
mul(3, 4): 12
Sum via reduce + operator.add: 21
Sorted by score: [('Bob', 95), ('Alice', 88), ('Charlie', 72)]


## 5. `itertools`

`itertools` provides fast, memory-efficient iterator building blocks for looping.

In [5]:
import itertools

# count(): infinite counter, bounded here with islice
print("count from 10:", list(itertools.islice(itertools.count(10, 2), 5)))

# cycle(): repeat a sequence forever, bounded here with islice
print("cycle RGB:", list(itertools.islice(itertools.cycle(["R", "G", "B"]), 7)))

# chain(): flatten multiple iterables into one
print("chain:", list(itertools.chain([1, 2], [3, 4], [5])))

# permutations and combinations
print("permutations(AB C, 2):", list(itertools.permutations("ABC", 2)))
print("combinations(ABC, 2):", list(itertools.combinations("ABC", 2)))

# groupby(): group consecutive items sharing a key
data = [1, 1, 2, 2, 2, 3, 1, 1]
grouped = [(key, list(group)) for key, group in itertools.groupby(data)]
print("groupby:", grouped)

count from 10: [10, 12, 14, 16, 18]
cycle RGB: ['R', 'G', 'B', 'R', 'G', 'B', 'R']
chain: [1, 2, 3, 4, 5]
permutations(AB C, 2): [('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]
combinations(ABC, 2): [('A', 'B'), ('A', 'C'), ('B', 'C')]
groupby: [(1, [1, 1]), (2, [2, 2, 2]), (3, [3]), (1, [1, 1])]


## 6. List Comprehensions & Nested List Comprehensions

Comprehensions are a concise, functional-style way to build a new list from an iterable, an
alternative to `map`/`filter` with a `for` loop.

In [6]:
# Basic comprehension
squares = [x * x for x in range(1, 6)]
print("Squares:", squares)

# Comprehension with a condition
even_squares = [x * x for x in range(1, 11) if x % 2 == 0]
print("Even squares:", even_squares)

# Nested comprehension: flatten a matrix
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flattened = [value for row in matrix for value in row]
print("Flattened matrix:", flattened)

# Nested comprehension: build a matrix (multiplication table)
mult_table = [[i * j for j in range(1, 4)] for i in range(1, 4)]
print("Multiplication table:", mult_table)

Squares: [1, 4, 9, 16, 25]
Even squares: [4, 16, 36, 64, 100]
Flattened matrix: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Multiplication table: [[1, 2, 3], [2, 4, 6], [3, 6, 9]]


## 7. Generators & Generator Expressions

Generators produce values lazily (one at a time, on demand) instead of building the whole result
in memory up front.

In [7]:
def even_numbers(limit):
    for i in range(limit):
        if i % 2 == 0:
            yield i

print("Generator function output:", list(even_numbers(10)))

# Generator expression, consumed directly by sum() without building a list
sum_of_squares = sum(x * x for x in range(1, 6))
print("Sum of squares via generator expression:", sum_of_squares)

Generator function output: [0, 2, 4, 6, 8]
Sum of squares via generator expression: 55


## 8. Decorators

A decorator is a function that wraps another function to add behavior before/after it runs,
without changing the original function's code.

In [8]:
def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@log_call
def add(a, b):
    return a + b

add(3, 5)

Calling add with args=(3, 5), kwargs={}
add returned 8


8

## 9. Looping Techniques

`enumerate()` and `zip()` are common functional-style helpers for looping over sequences without
manual index bookkeeping.

In [9]:
fruits = ["apple", "banana", "cherry"]

# enumerate(): loop with index and value together
for index, fruit in enumerate(fruits, start=1):
    print(index, fruit)

# zip(): loop over multiple sequences in parallel
prices = [1.5, 0.5, 3.0]
for fruit, price in zip(fruits, prices):
    print(f"{fruit}: ${price}")

# looping over a dict's items()
inventory = {"apple": 10, "banana": 5, "cherry": 20}
for name, count in inventory.items():
    print(f"{name} -> {count}")

1 apple
2 banana
3 cherry
apple: $1.5
banana: $0.5
cherry: $3.0
apple -> 10
banana -> 5
cherry -> 20
